<a href="https://colab.research.google.com/github/VinishReddyK/zipcast-epub/blob/main/notebooks/colab_zipcast_qwen3tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# zipcast: epub → audiobook (Qwen3-TTS)

This notebook does the entire conversion -- epub parsing and TTS -- on a free Colab GPU.

**Steps to use this notebook:**
1. Set **Runtime > Change runtime type > T4 GPU**.
2. Drag one or more `.epub` files into the Colab **Files** panel (left sidebar) so
   they land in `/content`.
3. Run every cell top to bottom. Every `.epub` found in `/content` is converted to
   a `.m4b`, one after another, and the last cell downloads them all.


In [ ]:
import os

# reduce CUDA memory fragmentation on long multi-chapter runs (must be set
# before torch initializes a CUDA context)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL = "https://github.com/VinishReddyK/zipcast-epub.git"  # @param {type:"string"}
REPO_DIR = "/content/zipcast"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
# install ffmpeg (for m4b muxing) and the epub-parsing + Qwen3-TTS python dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q -r {REPO_DIR}/colab/requirements.txt


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. go to Runtime > Change runtime type > T4 GPU, "
          "otherwise synthesis will be very slow.")


## Find the epub files

Scans `/content` for every `.epub` you've uploaded. No zip or pre-processing step --
just drop the raw epub files in.


In [ ]:
from colab.generate import find_epubs

epub_paths = find_epubs("/content")
print(f"found {len(epub_paths)} epub file(s):")
for p in epub_paths:
    print(f"  {p.name}")


## Convert all of them, one after another

Loads Qwen3-TTS once, then for each epub: parses it, synthesizes every chapter
(resumable -- re-running this cell skips chapters whose `.wav` already exists
under `/content/zipcast_work/<book>/wav`, and skips books whose `.m4b` already
exists in `/content`), and muxes the result into a single `.m4b` with chapter
markers and cover art via ffmpeg.


In [ ]:
SPEAKER = "ryan"  # @param ["ryan", "vivian", "sunny", "aria", "bella", "nova", "echo", "finn", "atlas"]
MODEL_NAME = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"  # @param {type:"string"}
BATCH_SIZE = 4  # @param {type:"integer"}
# chunks synthesized per model call. lower this (e.g. to 1-2) if you still hit
# CUDA out-of-memory errors on unusually long chapters.

from colab.generate import run_batch

output_paths = run_batch(
    content_dir="/content",
    speaker=SPEAKER.lower(),
    model_name=MODEL_NAME,
    device=device,
    batch_size=BATCH_SIZE,
)
print("\naudiobooks ready:")
for p in output_paths:
    print(f"  {p}")


In [ ]:
from google.colab import files

for p in output_paths:
    files.download(str(p))
